In [ ]:
# Load the Kedro IPython extension
%load_ext kedro.ipython

In [ ]:
df_base = catalog.load("second_stage_predictions").to_pandas()

In [ ]:
import dash
import numpy as np
import plotly.graph_objects as go
from dash import Input, Output, State, dash_table, dcc, html
from sklearn.preprocessing import StandardScaler

# Example df_base assumed to be defined externally
# df_base = pd.read_csv(...) or loaded beforehand

features = ["c_yearly_network_volumen"] + [
    f"share_customer_region_{i}" for i in range(1, 17)
]
region_cols = [
    col for col in df_base.columns if col.startswith("share_customer_region_")
]

# ------------- CONFIG: CUSTOM WEIGHTS ----------------
volume_weight = 1
region_share_weight = 1.0
feature_weights = np.array([volume_weight] + [region_share_weight] * len(region_cols))

# --- Dash App ---
app = dash.Dash(__name__)
server = app.server

app.layout = html.Div(
    [
        html.H2("KNN Explorer – Similar Clients by Region Share"),
        html.Div(
            [
                html.Label("Volume:"),
                dcc.Input(
                    id="input-volume",
                    type="number",
                    min=0,
                    step=1,
                    placeholder="Enter volume...",
                ),
                html.Label("Number of Neighbors (k):"),
                dcc.Input(id="input-k", type="number", value=5, step=1),
            ],
            style={"display": "flex", "gap": "20px", "marginBottom": "20px"},
        ),
        html.Div(
            [
                html.Div(
                    [
                        html.Label(f"{col}:"),
                        dcc.Input(id=col, type="number", value=0, step=0.01),
                    ]
                )
                for col in region_cols
            ],
            style={
                "display": "grid",
                "gridTemplateColumns": "repeat(4, 1fr)",
                "gap": "10px",
            },
        ),
        dcc.Graph(id="knn-plot"),
        html.H4("Top Nearest Neighbors"),
        html.Label("Select Columns to Display:"),
        dcc.Dropdown(
            id="column-selector",
            options=[{"label": col, "value": col} for col in df_base.columns],
            value=[
                "customer_id",
                "c_yearly_network_volumen",
                "corrected_predicted_value",
                "Distance",
            ],
            multi=True,
        ),
        dash_table.DataTable(
            id="neighbors-table",
            columns=[],
            data=[],
            style_table={"overflowX": "auto"},
            style_cell={"textAlign": "left", "padding": "5px"},
        ),
    ]
)


@app.callback(
    Output("knn-plot", "figure"),
    Output("neighbors-table", "data"),
    Input("input-volume", "value"),
    Input("input-k", "value"),
    *[Input(col, "value") for col in region_cols],
)
def update_knn(volume, k, *region_values):
    if volume is None or k is None or k <= 0:
        return dash.no_update, []

    region_array = np.array(region_values, dtype=np.float64)
    region_sum = region_array.sum()
    region_array = (
        region_array / region_sum
        if region_sum > 0
        else np.ones_like(region_array) / len(region_array)
    )

    new_point = np.concatenate(([volume], region_array)).reshape(1, -1)
    features_full = ["c_yearly_network_volumen"] + region_cols
    df_clean = df_base.dropna(
        subset=features_full + ["corrected_predicted_value"]
    ).copy()

    # Standardize and apply weights
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_clean[features_full])
    user_scaled = scaler.transform(new_point)
    X_weighted = X_scaled * feature_weights
    user_weighted = user_scaled * feature_weights

    distances = np.linalg.norm(X_weighted - user_weighted, axis=1)
    df_clean["Distance"] = distances
    df_sorted = df_clean.sort_values("Distance").reset_index(drop=True)
    df_knn = df_sorted.head(k).copy()
    df_knn["Rank"] = np.arange(1, k + 1)

    # Violin plot
    min_val = df_knn["corrected_predicted_value"].min()
    max_val = df_knn["corrected_predicted_value"].max()
    mean_val = df_knn["corrected_predicted_value"].mean()
    median_val = df_knn["corrected_predicted_value"].median()

    # Create figure
    fig = go.Figure()

    # Add horizontal line (like a simplified box)
    fig.add_shape(
        type="line",
        x0=min_val,
        y0=0,
        x1=max_val,
        y1=0,
        line=dict(color="gray", width=6),
    )

    # Add markers for each stat
    fig.add_trace(
        go.Scatter(
            x=[min_val],
            y=[0],
            mode="markers+text",
            name="Min",
            marker=dict(color="red", size=12),
            text=["Min"],
            textposition="top center",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[max_val],
            y=[0],
            mode="markers+text",
            name="Max",
            marker=dict(color="red", size=12),
            text=["Max"],
            textposition="top center",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[mean_val],
            y=[0],
            mode="markers+text",
            name="Mean",
            marker=dict(color="blue", size=12),
            text=["Mean"],
            textposition="top center",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[median_val],
            y=[0],
            mode="markers+text",
            name="Median",
            marker=dict(color="green", size=12),
            text=["Median"],
            textposition="top center",
        )
    )

    # Update layout
    fig.update_layout(
        title="Corrected Margin Summary (Top-k Nearest Clients)",
        xaxis_title="Corrected Margin",
        yaxis=dict(visible=False),
        showlegend=False,
        height=300,
        margin=dict(t=50, b=30),
    )

    # Return figure and KNN records for dynamic table
    return fig, df_knn.to_dict("records")


@app.callback(
    Output("neighbors-table", "columns"),
    Input("column-selector", "value"),
    State("neighbors-table", "data"),
)
def update_table_columns(selected_columns, knn_data):
    if not selected_columns or not knn_data:
        return []

    columns = [{"name": col, "id": col} for col in selected_columns]
    return columns


if __name__ == "__main__":
    app.run(debug=True, use_reloader=False, port=8123)


In [ ]:
df_base.columns